In this notebook, we show how to use the hippo client, henry, to dynamically push products into hippo.

Importantly, once pushed, all knowledge of the particular cluster or environment that push the files is gone -- the files are in the distributed data lake.

In [ ]:
from os.path import join as opj
from henry import Henry
from henry.collection import CollectionInstance
import hippometa as hm 

client = Henry()

Where the files are on my system, before pushing to hippo, is of course system specific. But you can imagine this is the end of a pipeline run, or even that the maps produced are not written to disk prior to being pushed directly from memory to hippo.

In [ ]:
map_dir = '/scratch/gpfs/ACT/data/act_dr6/dr6.02/maps/published'
window_dir = '/scratch/gpfs/SIMONSOBS/lat-iso/act_dr6/dr6.02/windows/published'
beam_dir = '/scratch/gpfs/SIMONSOBS/lat-iso/beams/dr6/main_beams'

The hippo API maps directly onto how typical users would arrange data on-disk: collections which can hold products and other collections. We can simply loop over arrays, grab each product from disk, insert into a hippo product with associated metadata, and finally arrange the hippo products into collections. The hippo idioms have minimal overhead.

Some products, like maps, are implemented as "mapsets": sort of mini collections, with a frozen template structure, of files that should always stick together (for instance a regular map with a source-free map). Other poducts, like beams, only contain one file. Particular files, or sources, are attached to "slugs" which are essentially attributes of a hippo product.

In [ ]:
# just do mid-frequencies
arrays = ['pa5', 'pa6']
freqs = ['f090', 'f150']

# mapset info
splits = range(4)
mapset_fn_template = 'act_dr6.02_std_AA_night_{array}_{freq}_4way_set{split}_{slug}.fits'

# window info
window_fn_template = 'window_dr6_{array}_{freq}_baseline.fits'

# beam info
beam_fn_template = 'coadd_{array}_{freq}_night_beam_tform_jitter_cmb.txt'

That's all we need. Let's loop over our products.

In [ ]:
top_collection = client.new_collection(
    name='Hippo ACT DR6 Demonstration',
    description='A collection of mid-frequency, baseline cosmology ACT DR6 maps, ' \
    'windows, and their beams. This is just enough to make pretty, reasonably ' \
    'accurate ACT DR6 power spectra. This particular collection is meant to ' \
    'demonstrate in a lightweight way the key features of distributed data ' \
    'delivery with hippo, and how to use its python API, henry.',
    products=[],
)

mapset_collection = client.new_collection(
    name='Hippo ACT DR6 Demonstration Maps',
    description='ACT DR6 cosmology-grade split maps at f090 and f150.',
    products=[],
)

# NOTE: mapsets also have a "mask" slug, so we could have also attached these files
# to the mapset products. But this demonstrates a bit more the workings of the data 
# lake
window_collection = client.new_collection(
    name='Hippo ACT DR6 Demonstration Windows',
    description='Apodized analysis windows for ACT DR6 cosmology-grade split maps at f090 and f150.',
    products=[],
)

beam_collection = client.new_collection(
    name='Hippo ACT DR6 Demonstration Beams',
    description='Isotropic instrument response functions for ACT DR6 cosmology-grade split maps at f090 and f150.',
    products=[],
)

# now the loop
for array in arrays:
    for freq in freqs:

        # we'll grab all the split maps into a collection of mapsets 
        mapset_array_freq_collection = client.new_collection(
            name=f'ACT DR6 Cosmology-grade Split Map ({array}, {freq}, all splits)',
            description=f'ACT DR6 cosmology-grade split maps for {array}, {freq}. Both nominal ' + \
            'and source-free versions are bundled together.',
            products=[],
            )

        for s in splits:

            # now we make the actual mapset
            mapset = client.new_product(
                name=f'ACT DR6 Cosmology-grade Split Map ({array}, {freq}, split {s})',
                description=f'All map data from 2017-2022 for array {array} in the {freq} band. Uses ' + \
                'improved bilinear pointing matrix and atmospheric detector relative calibrations to ' + \
                'reduce large-scale loss of power (transfer function) and low-ell correlated polarization ' + \
                'noise. Same maps used in the nominal ACT DR6 cosmology results of Louis et. al. 2025 and ' + \
                'Calabrese et. al. 2025.',
                metadata=hm.MapSet(telescope='act', instrument=array,
                                   frequency=freq, release='act-dr6.02',
                                   patch='AA', polarization_convention='COSMO', 
                                   pixelisation='cartesian', split=s),
                sources={'map': {'path': opj(map_dir, mapset_fn_template.format(array=array, freq=freq, split=s, slug='map')),
                                 'description': 'Nominal map (incudes point sources)'},
                         'source_free': {'path': opj(map_dir, mapset_fn_template.format(array=array, freq=freq, split=s, slug='map_srcfree')),
                                         'description': 'Map with point-sources fitted and subtracted.'}        
                        }
                )

            # and add it to the collection for this array and freq
            mapset_array_freq_collection.append(mapset)
        
        mapset_collection.append(mapset_array_freq_collection) # add a collection to a collection!
            
        # the windows and beams don't vary over split, so we can just make the 
        # products at this level (and add to their respective collections)
        window = client.new_product(
            name=f'Apodized Analysis Window for ACT DR6 Cosmology-grade Split Map ({array}, {freq}, any split)',
            description=rf'Analysis region used in the nominal ACT DR6 cosmology results of Louis et. al. 2025. ' + \
                'Roughly consists of threshold on map white noise level, crosslinking, and Planck 70% Galactic sky mask.',
            metadata=hm.MapSet(telescope='act', instrument=array,
                                frequency=freq, release='act-dr6.02',
                                patch='AA', pixelisation='cartesian'),
            sources={'map': {'path': opj(window_dir, window_fn_template.format(array=array, freq=freq)),
                             'description': 'Analysis window'},       
                    }
            )
        window_collection.append(window)

        beam = client.new_product(
            name=f'Main Beam for ACT DR6 Cosmology-grade Split Map ({array}, {freq}, any split)',
            description=rf'Isotropic instrument response function used in the nominal ACT DR6 cosmology results of Louis et. al. 2025.',
            metadata=hm.BeamMetadata(telescope='act', instrument=array, frequency=freq),
            sources={'data': {'path': opj(beam_dir, beam_fn_template.format(array=array, freq=freq)),
                             'description': '1d Harmonic transform'},       
                    }
            )
        beam_collection.append(beam)

top_collection.append(mapset_collection)
top_collection.append(window_collection)
top_collection.append(beam_collection)

Now we can push the top-level collection, and all its sub collections and sub products, to hippo. This involves sending large data volumes over the internet, so it may be slow.

In [ ]:
client.push(top_collection)